In [1]:
import os
import sys
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

c:\Users\VEDANT\manufacturing-predictive-maintenance\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sys.path.append(os.path.abspath("../src"))
from models.explainability import IndustrialExplainabilityEngine, generate_global_shap_artifacts

In [3]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [4]:
generate_global_shap_artifacts()

[1/4] Initializing Explainability Engine & Ingesting Splits...
[2/4] Computing SHAP Explanation Matrix on Test Holdout...
[3/4] Generating Global Summary and Importance Plots...
[4/4] Artifacts successfully serialized:
  -> Summary Plot:     c:\Users\VEDANT\manufacturing-predictive-maintenance\reports\figures\shap_summary_plot.png
  -> Feature Bar Plot: c:\Users\VEDANT\manufacturing-predictive-maintenance\reports\figures\shap_feature_importance.png
  -> Importance CSV:   c:\Users\VEDANT\manufacturing-predictive-maintenance\models\feature_importance_shap.csv


In [6]:
df_importance = pd.read_csv("../models/feature_importance_shap.csv")
df_importance.head(10)

,feature,mean_abs_shap
0,vibration_roll_mean_12h,2.622895
1,vibration_roll_mean_24h,2.591998
2,vibration_roll_mean_3h,2.531343
3,vibration,2.466533
4,pressure_roll_mean_24h,2.439884
5,pressure,2.154815
6,pressure_peak_to_avg_24h,1.806264
7,pressure_roll_mean_3h,1.417597
8,pressure_roll_mean_12h,1.123680
9,temperature_roll_mean_12h,1.019539


In [7]:
engine = IndustrialExplainabilityEngine()
test_df = pd.read_parquet("../data/processed/features/features_test.parquet")
X_test = test_df[engine.feature_cols]

In [8]:
shap_exp, shap_vals = engine.compute_shap_values(X_test)

plt.figure(figsize=(12, 6))
shap.plots.beeswarm(shap_exp, max_display=10, show=False)
plt.title("SHAP Beeswarm: Impact of Feature Value on Failure Prediction Probability", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

C:\Users\VEDANT\AppData\Local\Temp\ipykernel_28024\1484434258.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
features_all = pd.read_parquet("../data/processed/features/features_all.parquet")
m04_diag = engine.get_local_machine_attribution("M04", features_all, top_n=5)

print(f"Machine: M04 (Hydraulic Press) | Base Expected Value: {m04_diag['base_expected_value']:.3f} | Model Margin: {m04_diag['model_prediction_margin']:.3f}")
print("\nTop Root-Cause Drivers:")
for d in m04_diag["top_drivers"]:
    print(f"  • {d['feature']:<28} = {d['feature_value']:>8.2f} | SHAP: {d['shap_attribution']:>+7.4f} ({d['direction']})")

Machine: M04 (Hydraulic Press) | Base Expected Value: -28.222 | Model Margin: 232.517

Top Root-Cause Drivers:
  • temperature_roll_mean_12h    =    58.23 | SHAP: +111.1852 (Risk Increasing)
  • temperature_roll_mean_3h     =    55.17 | SHAP: +99.9201 (Risk Increasing)
  • temperature                  =    54.67 | SHAP: +87.4804 (Risk Increasing)
  • temperature_roll_mean_24h    =    57.60 | SHAP: +79.7497 (Risk Increasing)
  • pressure_roll_mean_24h       =   137.38 | SHAP: +46.5100 (Risk Increasing)


In [10]:
plt.figure(figsize=(10, 5))
shap.plots.waterfall(m04_diag["shap_explanation_object"], max_display=7, show=False)
plt.title("Local SHAP Waterfall Explanation: Machine M04 (Hydraulic Press)", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

C:\Users\VEDANT\AppData\Local\Temp\ipykernel_28024\3303551638.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
m08_diag = engine.get_local_machine_attribution("M08", features_all, top_n=5)

plt.figure(figsize=(10, 5))
shap.plots.waterfall(m08_diag["shap_explanation_object"], max_display=7, show=False)
plt.title("Local SHAP Waterfall Explanation: Machine M08 (Electric Motor - Nominal State)", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

C:\Users\VEDANT\AppData\Local\Temp\ipykernel_28024\1507161334.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
